# ML-03 - Frame Your Lane as an ML Task

**Selected lane: Lane 2 - Refresh / Content Opportunity Scoring**

This notebook frames the problem before model training. The goal is to help content teams decide which existing pages deserve human review first.

The output will be a prioritized review queue. It is decision support for editors and SEO specialists, not an automatic content-changing system.


## 1. My lane as an ML task (type)

**Task type: Ranking / scoring.**

The decision is: Which existing content pages should a content strategist, SEO specialist, or editor review first?

The intended output is a ranked list of pages with a priority score. Reviewers can then decide whether to refresh, expand, protect, monitor, or leave a page unchanged.

A ranking task fits better than a simple yes/no classification because the team needs to know which pages deserve attention first.

**Cost of a wrong call:**
- False positive: An editor spends time reviewing a page that does not need changes.
- False negative: A page that deserves attention may be overlooked.

The system will support human decisions. It will not automatically change content.


In [1]:
# Verify the dataset and record the task framing.
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = next(
    (
        parent / "data" / "raw" / "content_refresh_anonymized.csv"
        for parent in [Path.cwd(), *Path.cwd().parents]
        if (parent / "data" / "raw" / "content_refresh_anonymized.csv").exists()
    ),
    None
)

if DATA_PATH is None:
    raise FileNotFoundError("Dataset not found in the project data/raw folder.")

df = pd.read_csv(DATA_PATH)

print("DATASET CHECK")
print("-------------")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Task type: Ranking / scoring")
print("Output: Prioritized content review queue")
print("Decision user: Content strategists, SEO specialists, and editors")

assert len(df) == 30000, f"Expected 30,000 rows, found {len(df)}"
assert len(df.columns) == 44, f"Expected 44 columns, found {len(df.columns)}"
assert "content_id" in df.columns
print("Dataset structure check: PASS")


DATASET CHECK
-------------
Rows: 30000
Columns: 44
Task type: Ranking / scoring
Output: Prioritized content review queue
Decision user: Content strategists, SEO specialists, and editors
Dataset structure check: PASS


## 2. Target or proxy

**What will the task produce?**

The intended output is a priority score that ranks pages for human review.

The starter dataset also contains a current-snapshot proxy label:

`is_declining_proxy = 1` when `trend_direction == "down"`, otherwise `0`.

This label is derived from the comparison of recent and previous 30-day impressions. It is a rule-derived snapshot indicator, not an independently observed future outcome.

Therefore:
- It can be used to understand the available proxy label in this starter dataset.
- It must not be used as a model input feature.
- `trend_pct` must not be used as a model input feature.
- The trend comparison columns used to create the label must not be used to predict that same label.
- A genuine future-decline target would require features from an earlier period and an outcome measured in a later period.

**Current limitation:** This starter snapshot does not establish a verified future outcome. I will not claim that it predicts future decline.


In [2]:
# Inspect the available proxy label and its source.
assert "trend_direction" in df.columns
assert "trend_pct" in df.columns

df["is_declining_proxy"] = (
    df["trend_direction"].eq("down").astype(int)
)

proxy_count = int(df["is_declining_proxy"].sum())
proxy_rate = df["is_declining_proxy"].mean() * 100

print("CURRENT-SNAPSHOT PROXY CHECK")
print("----------------------------")
print("Rows marked down:", f"{proxy_count:,}")
print("Proxy positive rate:", f"{proxy_rate:.2f}%")
print("Proxy source: trend_direction == 'down'")
print("Future outcome independently observed: NO")
print("Suitable as a verified future-decline target: NO")

print("\nProxy label distribution:")
print(df["trend_direction"].value_counts(dropna=False))

print("\nThis proxy is for transparent starter-data analysis only.")


CURRENT-SNAPSHOT PROXY CHECK
----------------------------
Rows marked down: 16,262
Proxy positive rate: 54.21%
Proxy source: trend_direction == 'down'
Future outcome independently observed: NO
Suitable as a verified future-decline target: NO

Proxy label distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

This proxy is for transparent starter-data analysis only.


## 3. Success metric

**Primary metric: Precision@K.**

Precision@K measures how many of the first K pages in a ranked review queue are relevant according to the defined outcome.

For example, Precision@50 is:

`Relevant pages among the top 50 ranked pages / 50`

A higher value means that more of the pages selected for the review queue meet the chosen relevance definition.

For a genuine future-decline task, relevance must be determined from an independently observed future outcome. That outcome is not available in the current starter snapshot.

The current proxy can be used for a clearly labelled demonstration, but proxy Precision@K must not be presented as proof of future predictive performance.

The ranking will later be compared with a transparent rule-based baseline. A machine-learning model will only be useful if it demonstrates measurable improvement over that baseline.


In [3]:
# Define Precision@K and inspect the available proxy base rate.
def precision_at_k(ranked_data, target_column, k):
    if k <= 0:
        raise ValueError("k must be greater than zero")
    if len(ranked_data) < k:
        raise ValueError("The ranked dataset must contain at least k rows")

    top_k = ranked_data.head(k)
    return top_k[target_column].sum() / k

K = 50
proxy_base_rate = df["is_declining_proxy"].mean()

print("SUCCESS METRIC")
print("--------------")
print("Primary metric: Precision@K")
print("Example K:", K)
print(f"Current proxy base rate: {proxy_base_rate:.3f}")
print(f"Expected Precision@{K} for random ranking against this proxy: {proxy_base_rate:.3f}")
print("A real ranking score is not evaluated in this notebook.")
print("Future-outcome Precision@K: not currently measurable from this snapshot.")


SUCCESS METRIC
--------------
Primary metric: Precision@K
Example K: 50
Current proxy base rate: 0.542
Expected Precision@50 for random ranking against this proxy: 0.542
A real ranking score is not evaluated in this notebook.
Future-outcome Precision@K: not currently measurable from this snapshot.


## 4. The unit of analysis, as a real dataframe

**One row represents one pseudonymized content item (page).**

The dataset contains 30,000 content records across 32 pseudonymized clients.

Each page is the unit that receives a priority score and can appear once in the review queue.

`content_id` and `client_id` are identifiers for grouping, joining, and validation. They must not be used as model features.

A zero value in `avg_position` means no position data, not a rank of zero. It will be treated as missing when preparing data.


In [4]:
# Show the actual page-level dataframe.
print("DATAFRAME CHECK")
print("---------------")
print("Shape:", df.shape)
print("Unique content IDs:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())
print("One row represents: one pseudonymized content item")

# Treat avg_position == 0 as missing/no position data.
df["avg_position"] = pd.to_numeric(df["avg_position"], errors="coerce")
df.loc[df["avg_position"] == 0, "avg_position"] = np.nan

print("\nRows with missing average position after zero handling:",
      int(df["avg_position"].isna().sum()))

display_columns = [
    "content_id",
    "content_type",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "avg_position"
]

print("\nFirst five page records:")
display(df[display_columns].head())

assert df["content_id"].is_unique, "Expected one row per content item."
print("Page-level uniqueness check: PASS")


DATAFRAME CHECK
---------------
Shape: (30000, 45)
Unique content IDs: 30000
Unique clients: 32
One row represents: one pseudonymized content item

Rows with missing average position after zero handling: 1205

First five page records:


,content_id,content_type,impressions_90d,clicks_90d,sessions_90d,content_age_days,days_since_last_update,avg_position
0,content_304f48230142,keyword article,3803,29,17,187,20,10.6
1,content_a1fb4e703a9e,keyword article,15320,7,9,445,25,20.3
2,content_9aa793d4d895,keyword article,12581,11,11,141,20,36.5
3,content_331d6c4de07b,keyword article,11751,58,78,463,22,6.2
4,content_d99b7a2d90ca,keyword article,19140,24,145,263,14,44.0


Page-level uniqueness check: PASS


## 5. Why ML beats a fixed rule here

A single fixed rule may not represent every page fairly. Review priority can depend on several signals together, such as:

- Search impressions and clicks.
- Sessions and engagement.
- Content age and time since last update.
- Average search position.
- Content length and content type.
- Keyword context when it is available.

The relationship between these signals may differ across content types. Missing keyword data and missing word counts are also systematic in this dataset.

A model may help combine multiple signals and identify patterns that are difficult to express with one simple threshold.

**However, ML has not yet been proven better.** It must be compared with a transparent rule-based baseline using an appropriate validation design. If it does not improve the decision, a simpler rule is preferable.

The current 90-day snapshot is not enough to establish a leakage-safe future prediction setup. Feature and outcome windows must be aligned before future-outcome modeling.


In [5]:
# Inspect missingness by content type and list possible signal groups.
missingness_columns = [
    column for column in [
        "search_volume",
        "competition",
        "word_count",
        "main_intent",
        "avg_position"
    ]
    if column in df.columns
]

missingness_by_type = (
    df.groupby("content_type")[missingness_columns]
      .apply(lambda group: group.isna().mean() * 100)
      .round(1)
)

print("MISSINGNESS BY CONTENT TYPE (%)")
print("--------------------------------")
display(missingness_by_type)

signal_groups = {
    "Search performance": [
        "impressions_90d", "clicks_90d", "avg_position"
    ],
    "Engagement": [
        "sessions_90d", "engaged_sessions_90d", "engagement_rate"
    ],
    "Content freshness": [
        "content_age_days", "days_since_last_update"
    ],
    "Content properties": [
        "word_count", "char_count", "content_type"
    ],
    "Keyword context": [
        "search_volume", "competition", "main_intent"
    ]
}

print("\nPOTENTIAL SIGNAL GROUPS")
print("-----------------------")
for group, columns in signal_groups.items():
    available = [column for column in columns if column in df.columns]
    print(f"{group}: {', '.join(available)}")

print("\nLeakage and identifier exclusions:")
print("- content_id and client_id: identifiers only")
print("- trend_direction and trend_pct: label-derived fields")
print("- Recent/previous 30-day trend inputs: do not use to predict the proxy derived from them")
print("- Provider/model metadata: not model features for this task")

print("\nML value still needs to be tested against a rule-based baseline.")


MISSINGNESS BY CONTENT TYPE (%)
--------------------------------


,search_volume,competition,word_count,main_intent,avg_position
content_type,,,,,
comparison article,0.0,0.0,0.0,0.0,0.0
feedly article,100.0,100.0,0.0,100.0,34.8
keyword article,1.4,1.4,28.3,1.0,1.7



POTENTIAL SIGNAL GROUPS
-----------------------
Search performance: impressions_90d, clicks_90d, avg_position
Engagement: sessions_90d, engaged_sessions_90d, engagement_rate
Content freshness: content_age_days, days_since_last_update
Content properties: word_count, char_count, content_type
Keyword context: search_volume, competition, main_intent

Leakage and identifier exclusions:
- content_id and client_id: identifiers only
- trend_direction and trend_pct: label-derived fields
- Recent/previous 30-day trend inputs: do not use to predict the proxy derived from them
- Provider/model metadata: not model features for this task

ML value still needs to be tested against a rule-based baseline.


## Final task frame

For content strategists, SEO specialists, and editors deciding which existing pages to review first, I will develop a ranked content-review queue using page-level search, engagement, freshness, and content signals.

The intended output is a priority score, evaluated with Precision@K when an appropriate outcome is available. The starter dataset contains a rule-derived current-snapshot decline proxy, not a verified future outcome. A wrong ranking can waste editorial time or cause an important page to be overlooked. Multiple signals may justify testing ML, but ML must demonstrate improvement over a transparent rule-based baseline. Results will be described as decision support, not causal proof or guaranteed future prediction.

## Self-check

- [x] Selected lane and task type are consistent with Week 1.
- [x] Decision, users, action, and error costs are specified.
- [x] Current proxy is distinguished from a future observed outcome.
- [x] Precision@K is defined with its limitations.
- [x] Unit of analysis is shown as a real dataframe.
- [x] Identifier and leakage exclusions are documented.
- [x] ML is not claimed to be better before baseline comparison.
- [ ] Run the notebook from top to bottom and inspect all outputs.
- [ ] Review the notebook for public-safe content before committing.
